In [1]:
# | default_exp gsc_storage


# GSC Storage
> Query, filter, and persist Google Search Console analytics data in SQLite.

In [1]:
# | export
from sqlmodel import Session, select
from seo_rat.models import GSCAnalytics
from datetime import datetime, timedelta
from functools import partial
from fastcore.basics import compose
from sqlalchemy import func, not_, or_
from seo_rat.sqlite_db import  get_session
from seo_rat.models import GSCAnalytics
from seo_rat.gsc_client import GSCAuth
from seo_rat.gsc_client import get_date_range

In [2]:
# | export
def filter_site(query,  # SQLModel query
                site_url: str  # GSC property URL
                ):
    "Filter query by site URL."
    return query.where(GSCAnalytics.site_url == site_url)


def filter_dates(query,  # SQLModel query
                 start: str,  # Start date (YYYY-MM-DD)
                 end: str  # End date (YYYY-MM-DD)
                 ):
    "Filter query by date range."
    return query.where(GSCAnalytics.date.between(start, end))


def filter_dimension(query,  # SQLModel query
                     dimension: str,  # GSCAnalytics field name
                     value: str  # Value to match
                     ):
    "Filter query by a specific dimension value."
    return query.where(getattr(GSCAnalytics, dimension) == value)


def filter_exclude_pages(query,  # SQLModel query
                         exclude_pages: list[str]  # Page path substrings to exclude
                         ):
    "Exclude rows where page contains any of the given substrings."
    return query.where(not_(or_(*[GSCAnalytics.page.contains(p) for p in exclude_pages])))


In [3]:
# | export
from sqlmodel import SQLModel


class AnalyticsSummary(SQLModel):
    query: str
    clicks: int
    impressions: int

In [4]:
#| export 
from urllib.parse import unquote


def normalize_url(url: str) -> str:
    """Normalize URL by decoding percent-encoding and standardizing separators"""
    url = unquote(url)
    url = url.replace("-", " ").replace("_", " ")
    return url


In [5]:
# | export
def parse_gsc_row(row:dict,      # Raw GSC API row
                  site_url:str,  # GSC property URL
                  date:str       # Date of the row (YYYY-MM-DD)
                 ) -> GSCAnalytics:
    "Parse a raw GSC API row into a GSCAnalytics instance."
    dims = dict(zip(("query", "page", "country", "device"), row["keys"]))
    return GSCAnalytics(site_url=site_url, date=date,
                        query=dims.get("query"), page=dims.get("page"),
                        country=dims.get("country"), device=dims.get("device"),
                        clicks=row["clicks"], impressions=row["impressions"],
                        ctr=row["ctr"], position=row["position"])


In [6]:
# | export
from sqlalchemy.dialects.sqlite import insert

def store_gsc_data(session:Session,  # Active database session
                   site_url:str,     # GSC property URL
                   date:str,         # Date of the data (YYYY-MM-DD)
                   rows:list[dict]   # Raw GSC API rows
                  ) -> None:
    "Store GSC rows with upsert (update on conflict)."
    for row in rows:
        record = parse_gsc_row(row, site_url, date)
        stmt = insert(GSCAnalytics).values(**record.model_dump(exclude={"id"}))
        stmt = stmt.on_conflict_do_update(
            index_elements=["site_url", "date", "query", "page", "country", "device"],
            set_={"clicks": record.clicks, "impressions": record.impressions,
                  "ctr": record.ctr, "position": record.position}
        )
        session.exec(stmt)
    session.commit()


In [7]:
# | hide
#| eval: false
from fastcore.test import test_eq
from seo_rat.gsc_client import GSCAuth, fetch_gsc_data, get_date_range
from sqlmodel import Session
from pprint import pprint

auth = GSCAuth()
start, end = get_date_range("last_days", days=30)
data = fetch_gsc_data(auth, "sc-domain:kareemai.com", start, end)


In [8]:
# | hide
#| eval: false
# with get_session() as sesssion:
    # store_gsc_data(session, "sc-domain:kareemai.com", start, data[:5])
    # # verfiy stored
    # test_eq(len(session.exec(select(GSCAnalytics)).all()) > 4, True)

In [9]:
# | export
def get_top_queries(session: Session,  # Active database session
                    site_url: str,  # GSC property URL
                    start_date: str,  # Start date (YYYY-MM-DD)
                    end_date: str,  # End date (YYYY-MM-DD)
                    country: str | None = None,  # Filter by country code
                    page_path: str | None = None,  # Filter by page path substring
                    limit: int = 10,  # Max rows to return
                    sort_by: str = "clicks"  # Sort by 'clicks' or 'impressions'
                    ) -> list[dict]:
    "Get top performing queries, optionally filtered by country and page."
    base = select(GSCAnalytics.query,
                  func.sum(GSCAnalytics.clicks).label("total_clicks"),
                  func.sum(GSCAnalytics.impressions).label("total_impressions"),
                  func.avg(GSCAnalytics.position).label("avg_position"),
                  func.avg(GSCAnalytics.ctr).label("avg_ctr"))
    filters = [partial(filter_site, site_url=site_url),
               partial(filter_dates, start=start_date, end=end_date)]
    if country: filters.append(partial(filter_dimension, dimension="country", value=country))
    if page_path: filters.append(lambda q: q.where(GSCAnalytics.page.contains(page_path)))
    sort_col = func.sum(GSCAnalytics.impressions) if sort_by == "impressions" else func.sum(GSCAnalytics.clicks)
    query = (compose(*filters)(base)
             .where(GSCAnalytics.query.isnot(None))
             .group_by(GSCAnalytics.query)
             .order_by(sort_col.desc())
             .limit(limit))
    return [row._asdict() for row in session.exec(query)]


In [10]:
#| export
def get_top_pages(session: Session, site_url: str, start_date: str, end_date: str,
                  country: str | None = None, limit: int = 20, sort_by: str = "clicks") -> list[dict]:
    "Get top performing pages by clicks or impressions."
    base = select(GSCAnalytics.page,
                  func.sum(GSCAnalytics.clicks).label("total_clicks"),
                  func.sum(GSCAnalytics.impressions).label("total_impressions"),
                  func.avg(GSCAnalytics.position).label("avg_position"),
                  func.avg(GSCAnalytics.ctr).label("avg_ctr"))
    filters = [partial(filter_site, site_url=site_url),
               partial(filter_dates, start=start_date, end=end_date)]
    if country: filters.append(partial(filter_dimension, dimension="country", value=country))
    sort_col = func.sum(GSCAnalytics.impressions) if sort_by == "impressions" else func.sum(GSCAnalytics.clicks)
    query = (compose(*filters)(base)
             .where(GSCAnalytics.page.isnot(None))
             .group_by(GSCAnalytics.page)
             .order_by(sort_col.desc())
             .limit(limit))
    return [row._asdict() for row in session.exec(query)]


In [11]:
#| export
def get_wins(session: Session, site_url: str, start_date: str, end_date: str,
             min_impressions: int = 100, min_position: float = 10.0,
             max_position: float = 50.0, country: str | None = None,
             page_url: str | None = None, limit: int = 20) -> list[dict]:
    "Get high-impression, low-ranking keyword opportunities."
    base = select(GSCAnalytics.query,
                  func.sum(GSCAnalytics.clicks).label("total_clicks"),
                  func.sum(GSCAnalytics.impressions).label("total_impressions"),
                  func.avg(GSCAnalytics.position).label("avg_position"),
                  func.avg(GSCAnalytics.ctr).label("avg_ctr"))
    filters = [partial(filter_site, site_url=site_url),
               partial(filter_dates, start=start_date, end=end_date)]
    if country: filters.append(partial(filter_dimension, dimension="country", value=country))
    if page_url: filters.append(lambda q: q.where(GSCAnalytics.page == page_url))
    query = (compose(*filters)(base)
             .where(GSCAnalytics.query.isnot(None))
             .group_by(GSCAnalytics.query)
             .having(func.sum(GSCAnalytics.impressions) >= min_impressions,
                     func.avg(GSCAnalytics.position) >= min_position,
                     func.avg(GSCAnalytics.position) <= max_position)
             .order_by(func.sum(GSCAnalytics.impressions).desc())
             .limit(limit))
    return [row._asdict() for row in session.exec(query)]



In [ ]:
# | hide
#| eval: false

test_top_queries = get_top_queries(
session,
site_url = "sc-domain:kareemai.com",
start_date = start,
end_date = end,
country = "egy",
)
test_top_queries

[{'query': 'huawei freebuds 7i',
  'total_clicks': 3,
  'total_impressions': 562,
  'avg_position': 6.339126971243043,
  'avg_ctr': 0.007569444444444444},
 {'query': 'freebuds 7i',
  'total_clicks': 3,
  'total_impressions': 153,
  'avg_position': 7.554898777692895,
  'avg_ctr': 0.03725490196078431},
 {'query': 'proteinai',
  'total_clicks': 1,
  'total_impressions': 1,
  'avg_position': 9.0,
  'avg_ctr': 1.0},
 {'query': 'oneplus pad 3',
  'total_clicks': 1,
  'total_impressions': 110,
  'avg_position': 10.305416666666666,
  'avg_ctr': 0.008333333333333333},
 {'query': 'one plus pad 3',
  'total_clicks': 1,
  'total_impressions': 31,
  'avg_position': 11.283333333333333,
  'avg_ctr': 0.025},
 {'query': 'huawei earbuds 7i',
  'total_clicks': 1,
  'total_impressions': 45,
  'avg_position': 7.04004329004329,
  'avg_ctr': 0.045454545454545456},
 {'query': 'huawei buds 7i review',
  'total_clicks': 1,
  'total_impressions': 4,
  'avg_position': 1.6666666666666667,
  'avg_ctr': 0.1666666666

In [12]:
# | export
def get_top_queries_excluding_pages(session: Session,  # Active database session
                                    site_url: str,  # GSC property URL
                                    start_date: str,  # Start date (YYYY-MM-DD)
                                    end_date: str,  # End date (YYYY-MM-DD)
                                    exclude_pages: list[str],  # Page substrings to exclude
                                    country: str | None = None,  # Filter by country code
                                    limit: int = 10  # Max rows to return
                                    ) -> list[dict]:
    "Get top queries excluding specific pages."
    base = select(GSCAnalytics.query,
                  func.sum(GSCAnalytics.clicks).label("total_clicks"),
                  func.sum(GSCAnalytics.impressions).label("total_impressions"))
    filters = [partial(filter_site, site_url=site_url),
               partial(filter_dates, start=start_date, end=end_date),
               partial(filter_exclude_pages, exclude_pages=exclude_pages)]
    if country: filters.append(partial(filter_dimension, dimension="country", value=country))
    query = (compose(*filters)(base)
             .where(GSCAnalytics.query.isnot(None))
             .group_by(GSCAnalytics.query)
             .order_by(func.sum(GSCAnalytics.clicks).desc())
             .limit(limit))
    return [row._asdict() for row in session.exec(query)]


In [15]:
# | hide
#| eval: false
from sqlalchemy.testing.exclusions import exclude

test_exclude_page_from_top_queries = get_top_queries_excluding_pages(
session,
site_url = "sc-domain:kareemai.com",
start_date = start,
end_date = end,
exclude_pages = [
    "https://kareemai.com/blog/posts/products_reviews/Huawei%20freebuds%207i.html",
    "https://kareemai.com/blog/posts/products_reviews/Huawei%20freebuds%205i.html",
],
country = "egy",
)
pprint(test_exclude_page_from_top_queries)

[{'query': 'proteinai', 'total_clicks': 1, 'total_impressions': 1},
 {'query': 'oneplus pad 3', 'total_clicks': 1, 'total_impressions': 110},
 {'query': 'one plus pad 3', 'total_clicks': 1, 'total_impressions': 31},
 {'query': 'tarteel jobs', 'total_clicks': 0, 'total_impressions': 1},
 {'query': 'tarteel careers', 'total_clicks': 0, 'total_impressions': 1},
 {'query': 'tarteel ai jobs', 'total_clicks': 0, 'total_impressions': 1},
 {'query': 'tarteel ai careers', 'total_clicks': 0, 'total_impressions': 2},
 {'query': 'super productivity linux',
  'total_clicks': 0,
  'total_impressions': 1},
 {'query': 'super productivity app review 2026',
  'total_clicks': 0,
  'total_impressions': 4},
 {'query': 'super productivity app', 'total_clicks': 0, 'total_impressions': 1}]


In [13]:
# | export
def get_page_analytics(session: Session,  # Active database session
                       site_url: str,  # GSC property URL
                       page_path: str,  # Partial page path to match
                       start_date: str,  # Start date (YYYY-MM-DD)
                       end_date: str  # End date (YYYY-MM-DD)
                       ) -> dict:
    "Get aggregated analytics and top queries for a specific page."
    filters = compose(
        partial(filter_site, site_url=site_url),
        partial(filter_dates, start=start_date, end=end_date),
    )
    base = filters(select(
        func.sum(GSCAnalytics.clicks).label("total_clicks"),
        func.sum(GSCAnalytics.impressions).label("total_impressions"),
        func.avg(GSCAnalytics.position).label("avg_position"),
        func.avg(GSCAnalytics.ctr).label("avg_ctr"),
    )).where(GSCAnalytics.page.contains(page_path))
    row = session.exec(base).first()

    top_queries = session.exec(
        filters(select(GSCAnalytics.query, func.sum(GSCAnalytics.clicks).label("clicks"))
                .where(GSCAnalytics.page.contains(page_path), GSCAnalytics.query.isnot(None))
                .group_by(GSCAnalytics.query)
                .order_by(func.sum(GSCAnalytics.clicks).desc())
                .limit(10))
    ).all()

    return {"page_path": page_path,
            "total_clicks": row.total_clicks or 0,
            "total_impressions": row.total_impressions or 0,
            "avg_position": row.avg_position or 0,
            "avg_ctr": row.avg_ctr or 0,
            "top_queries": [q for q, _ in top_queries]}



In [17]:
# | hide
#| eval: false
test_page_analytics = get_page_analytics(
    session,
    site_url="sc-domain:kareemai.com",
    page_path="https://kareemai.com",
    start_date=start,
    end_date=end,
)
pprint(test_page_analytics)

{'avg_ctr': 0.010992418206696242,
 'avg_position': 6.237940895402041,
 'page_path': 'https://kareemai.com',
 'top_queries': ['huawei freebuds 7i review',
                 'huawei freebuds 7i',
                 'huawei freebuds 5i vs 7i',
                 'freebuds 7i',
                 'freebuds 5i vs 7i',
                 'huawei freebuds 5i review',
                 'freebuds 7i review',
                 'huawei freebuds 7i vs 5i',
                 'super productivity review',
                 'huawei freebuds 5i'],
 'total_clicks': 145,
 'total_impressions': 16988}


In [14]:
# | export
def get_analytics_by_date_range(session: Session,  # Active database session
                                site_url: str,  # GSC property URL
                                start_date: str,  # Start date (YYYY-MM-DD)
                                end_date: str  # End date (YYYY-MM-DD)
                                ) -> list[GSCAnalytics]:
    "Get all raw GSC analytics rows for a date range."
    return session.exec(
        compose(partial(filter_site, site_url=site_url),
                partial(filter_dates, start=start_date, end=end_date)
                )(select(GSCAnalytics))
    ).all()


In [19]:
# | hide
#| eval: false

test_get_analytics_by_date_range = get_analytics_by_date_range(
    session, site_url="sc-domain:kareemai.com", start_date=start, end_date=end
)
pprint(test_get_analytics_by_date_range)

[GSCAnalytics(query='"paraphrase-multilingual-minilm-l12-v2" arabic', id=390054, country='usa', clicks=0, ctr=0.0, created_at=datetime.datetime(2026, 4, 1, 7, 55, 33, 607072), page='https://kareemai.com/blog/posts/minishlab/blog_zaraah.html', date='2026-03-06', site_url='sc-domain:kareemai.com', device='TABLET', impressions=1, position=9.0),
 GSCAnalytics(query='7i freebuds', id=390055, country='usa', clicks=0, ctr=0.0, created_at=datetime.datetime(2026, 4, 1, 7, 55, 33, 608308), page='https://kareemai.com/blog/posts/products_reviews/Huawei%20freebuds%207i.html', date='2026-03-06', site_url='sc-domain:kareemai.com', device='DESKTOP', impressions=1, position=1.0),
 GSCAnalytics(query='arabic tips for engineers', id=390056, country='pak', clicks=0, ctr=0.0, created_at=datetime.datetime(2026, 4, 1, 7, 55, 33, 609661), page='https://kareemai.com/til/', date='2026-03-06', site_url='sc-domain:kareemai.com', device='DESKTOP', impressions=1, position=1.0),
 GSCAnalytics(query='arabic tokenizer

In [15]:
# | export
def get_trends(session: Session,  # Active database session
               site_url: str,  # GSC property URL
               start_date: str,  # Start date (YYYY-MM-DD)
               end_date: str,  # End date (YYYY-MM-DD)
               dimension: str | None = None  # Optional dimension to group by
               ) -> list[dict]:
    "Get click/impression trends over time, optionally grouped by a dimension."
    columns = [GSCAnalytics.date,
               func.sum(GSCAnalytics.clicks).label("clicks"),
               func.sum(GSCAnalytics.impressions).label("impressions"),
               func.avg(GSCAnalytics.position).label("avg_position"),
               func.avg(GSCAnalytics.ctr).label("avg_ctr")]
    group_by = [GSCAnalytics.date]
    if dimension:
        dim_col = getattr(GSCAnalytics, dimension)
        columns.insert(1, dim_col)
        group_by.append(dim_col)
    query = (compose(partial(filter_site, site_url=site_url),
                     partial(filter_dates, start=start_date, end=end_date))
             (select(*columns))
             .group_by(*group_by)
             .order_by(GSCAnalytics.date))
    return [row._asdict() for row in session.exec(query)]


In [21]:
# | hide
#| eval: false
test_get_trends = get_trends(
    session, site_url="sc-domain:kareemai.com", start_date=start, end_date=end
)
pprint(test_get_trends)

[{'avg_ctr': 0.00533605812897366,
  'avg_position': 6.079551113368197,
  'clicks': 4,
  'date': '2026-03-06',
  'impressions': 736},
 {'avg_ctr': 0.012008101851851851,
  'avg_position': 6.225755602904041,
  'clicks': 6,
  'date': '2026-03-07',
  'impressions': 724},
 {'avg_ctr': 0.011188271604938271,
  'avg_position': 6.411227389201423,
  'clicks': 8,
  'date': '2026-03-08',
  'impressions': 739},
 {'avg_ctr': 0.006425702811244979,
  'avg_position': 6.366732169318987,
  'clicks': 5,
  'date': '2026-03-09',
  'impressions': 741},
 {'avg_ctr': 0.015745098039215685,
  'avg_position': 6.425654550190273,
  'clicks': 9,
  'date': '2026-03-10',
  'impressions': 737},
 {'avg_ctr': 0.02169312169312169,
  'avg_position': 6.102397953233376,
  'clicks': 9,
  'date': '2026-03-11',
  'impressions': 725},
 {'avg_ctr': 0.016709511568123392,
  'avg_position': 5.958961396151461,
  'clicks': 9,
  'date': '2026-03-12',
  'impressions': 712},
 {'avg_ctr': 0.008407243163340723,
  'avg_position': 6.096935959

In [16]:
# | export
def get_analytics_by(session: Session,  # Active database session
                     site_url: str,  # GSC property URL
                     start_date: str,  # Start date (YYYY-MM-DD)
                     end_date: str,  # End date (YYYY-MM-DD)
                     dimension: str,  # GSCAnalytics field to filter by
                     value: str  # Value to match
                     ) -> list[AnalyticsSummary]:
    "Get query-level analytics filtered by a specific dimension value."
    base = select(GSCAnalytics.query,
                  func.sum(GSCAnalytics.clicks).label("clicks"),
                  func.sum(GSCAnalytics.impressions).label("impressions"))
    query = (compose(partial(filter_site, site_url=site_url),
                     partial(filter_dates, start=start_date, end=end_date),
                     partial(filter_dimension, dimension=dimension, value=value))
             (base).group_by(GSCAnalytics.query))
    return [AnalyticsSummary(**row._asdict()) for row in session.exec(query)]


In [23]:
# | hide
#| eval: false
test_result = get_analytics_by(
    session,
    site_url="sc-domain:kareemai.com",
    start_date=start,
    end_date=end,
    dimension="country",
    value="egy",
)

pprint(test_result)


[AnalyticsSummary(query='7i freebuds', clicks=0, impressions=2),
 AnalyticsSummary(query='arabert v2', clicks=0, impressions=2),
 AnalyticsSummary(query='arabic tokenizer', clicks=0, impressions=1),
 AnalyticsSummary(query='buds 7i huawei', clicks=0, impressions=2),
 AnalyticsSummary(query='cloud gpu pricing comparison', clicks=0, impressions=1),
 AnalyticsSummary(query='colgrep', clicks=0, impressions=2),
 AnalyticsSummary(query='dailyaiproductivity reviews 2026', clicks=0, impressions=1),
 AnalyticsSummary(query='does it have a pen', clicks=0, impressions=1),
 AnalyticsSummary(query='does the huawei matepad have google', clicks=0, impressions=1),
 AnalyticsSummary(query='frebuds 7i', clicks=0, impressions=1),
 AnalyticsSummary(query='free buds 7i', clicks=0, impressions=2),
 AnalyticsSummary(query='freebids 7i', clicks=0, impressions=1),
 AnalyticsSummary(query='freebuds 5i', clicks=0, impressions=2),
 AnalyticsSummary(query='freebuds 5i noise cancelling', clicks=0, impressions=1),
 

In [24]:
# | hide
#| eval: false
test_get_analytics_by_device = get_analytics_by(
    session,
    site_url="sc-domain:kareemai.com",
    start_date=start,
    end_date=end,
    dimension="device",
    value="MOBILE",
)
pprint(test_get_analytics_by_device)


[AnalyticsSummary(query='"vercel" "transfer" "whois privacy"', clicks=0, impressions=11),
 AnalyticsSummary(query='"vercel" "whois privacy" transfer', clicks=0, impressions=2),
 AnalyticsSummary(query='.', clicks=0, impressions=1),
 AnalyticsSummary(query='5 foto', clicks=0, impressions=2),
 AnalyticsSummary(query='5i huawei', clicks=0, impressions=1),
 AnalyticsSummary(query='5i huawei freebuds', clicks=0, impressions=2),
 AnalyticsSummary(query='5i vs 7i', clicks=0, impressions=1),
 AnalyticsSummary(query='7i freebuds', clicks=0, impressions=7),
 AnalyticsSummary(query='7i huawei', clicks=0, impressions=4),
 AnalyticsSummary(query='affordable gpu instances hivenet', clicks=0, impressions=2),
 AnalyticsSummary(query='affordable hivenet compute benchmark', clicks=0, impressions=2),
 AnalyticsSummary(query='affordable rent gpu hivenet', clicks=0, impressions=1),
 AnalyticsSummary(query='airpods huawei 7i', clicks=0, impressions=1),
 AnalyticsSummary(query='anker liberty 5 vs huawei free

In [17]:
# | export
from seo_rat.gsc_client import fetch_gsc_data
import time


def store_single_date(session:Session, # Active database session
                      auth:GSCAuth,    # Authenticated GSCAuth instance
                      site_url:str,    # GSC property URL
                      date:str         # Date to fetch and store (YYYY-MM-DD)
                     ) -> int:
    "Fetch and store GSC data for a single date. Returns number of records stored."
    rows = fetch_gsc_data(auth, site_url, date, date)
    store_gsc_data(session, site_url, date, rows)
    return len(rows)

In [ ]:
# | hide
# | test
#| eval: false
count = store_single_date(session, auth, "sc-domain:kareemai.com", start)
test_eq(count > 0, True)


In [ ]:
# | test
#| eval: false
# Check what's actually stored
stored = session.exec(select(GSCAnalytics).limit(1)).first()
print(f"Query: {stored.query}")
print(f"Clicks: {stored.clicks}")
print(f"Date: {stored.date}")
test_eq(stored.site_url, "sc-domain:kareemai.com")


Query: أنواع السباكة
Clicks: 0
Date: 2024-11-11


AssertionError: ==:
sc-domain:awazly.com
sc-domain:kareemai.com

In [18]:
#| export
def _sync_dates(session: Session,  # Active database session
                auth: GSCAuth,  # Authenticated GSCAuth instance
                site_url: str,  # GSC property URL
                dates: list[str]  # List of dates to sync (YYYY-MM-DD)
                ) -> dict:
    "Fetch and store GSC data for a list of dates, returning a results summary."
    results = {"successful": [], "failed": [], "total_records": 0}
    for date in dates:
        try:
            count = store_single_date(session, auth, site_url, date)
            results["successful"].append(date)
            results["total_records"] += count
            print(f"Synced {date}: {count} records")
        except Exception as e:
            results["failed"].append({"date": date, "error": str(e)})
            print(f"Failed {date}: {e}")
        time.sleep(1)
    return results


In [19]:
# | export
def store_date_range(session: Session, auth: GSCAuth, site_url: str, start_date: str, end_date: str) -> dict:
    "Fetch and store GSC data for all dates in range."
    return _sync_dates(session, auth, site_url, iter_dates(start_date, end_date))



In [ ]:
# | hide
#| eval: false
result = store_date_range(session, auth, "sc-domain:https://emdadelgaz.com/", start, end)
test_eq(len(result["successful"]), 1)
print(f"Stored {result['total_records']} records")


Processing 2026-02-27 (1/31)...
Processing 2026-02-28 (2/31)...
Processing 2026-03-01 (3/31)...
Processing 2026-03-02 (4/31)...
Processing 2026-03-03 (5/31)...
Processing 2026-03-04 (6/31)...
Processing 2026-03-05 (7/31)...
Processing 2026-03-06 (8/31)...
Processing 2026-03-07 (9/31)...
Processing 2026-03-08 (10/31)...
Processing 2026-03-09 (11/31)...
Processing 2026-03-10 (12/31)...
Processing 2026-03-11 (13/31)...
Processing 2026-03-12 (14/31)...
Processing 2026-03-13 (15/31)...
Processing 2026-03-14 (16/31)...
Processing 2026-03-15 (17/31)...
Processing 2026-03-16 (18/31)...
Processing 2026-03-17 (19/31)...
Processing 2026-03-18 (20/31)...
Processing 2026-03-19 (21/31)...
Processing 2026-03-20 (22/31)...
Processing 2026-03-21 (23/31)...
Processing 2026-03-22 (24/31)...
Processing 2026-03-23 (25/31)...
Processing 2026-03-24 (26/31)...
Processing 2026-03-25 (27/31)...
Processing 2026-03-26 (28/31)...
Processing 2026-03-27 (29/31)...
Processing 2026-03-28 (30/31)...
Processing 2026-03-

AssertionError: ==:
0
1

In [20]:
#| hide
#| eval: false
r = session.exec(select(GSCAnalytics).limit(1)).first()
print(f"Query: {r.query}")
print(f"Page: {r.page}")
print(f"Country: {r.country}")
print(f"Device: {r.device}")
print(f"Clicks: {r.clicks}")
print(f"Impressions: {r.impressions}")
print(f"CTR: {r.ctr}")
print(f"Position: {r.position}")


NameError: name 'session' is not defined

In [ ]:
#| hide
#| eval: false
top = get_top_queries(session, "sc-domain:emdadelgaz.com", start, start, limit=5)
for q in top:
    print(
        f"{q['query']}: {q['total_clicks']} clicks, {q['total_impressions']} impressions"
    )


In [ ]:
#| hide
from sqlalchemy import delete


In [ ]:
# | hide
#| eval: false

# with get_session() as session:

    # session.exec(delete(GSCAnalytics))
    # session.commit()


In [34]:
# | hide
#| eval: false

# delay 3 because GSC api limitation
end = datetime.now() - timedelta(days=3)
start = end - timedelta(days=365 * 2)

result = store_date_range(
    session,
    auth,
    "sc-domain:smaagarden.com",
    start.strftime("%Y-%m-%d"),
    end.strftime("%Y-%m-%d"),
)


Synced 2024-04-05: 0 records
Synced 2024-04-06: 0 records
Synced 2024-04-07: 0 records
Synced 2024-04-08: 0 records
Synced 2024-04-09: 0 records
Synced 2024-04-10: 0 records
Synced 2024-04-11: 0 records
Synced 2024-04-12: 0 records
Synced 2024-04-13: 0 records
Synced 2024-04-14: 0 records
Synced 2024-04-15: 0 records
Synced 2024-04-16: 0 records
Synced 2024-04-17: 0 records
Synced 2024-04-18: 0 records
Synced 2024-04-19: 0 records
Synced 2024-04-20: 0 records
Synced 2024-04-21: 0 records
Synced 2024-04-22: 0 records
Synced 2024-04-23: 0 records
Synced 2024-04-24: 0 records
Synced 2024-04-25: 0 records
Synced 2024-04-26: 0 records
Synced 2024-04-27: 0 records
Synced 2024-04-28: 0 records
Synced 2024-04-29: 0 records
Synced 2024-04-30: 0 records
Synced 2024-05-01: 0 records
Synced 2024-05-02: 0 records
Synced 2024-05-03: 0 records
Synced 2024-05-04: 0 records
Synced 2024-05-05: 0 records
Synced 2024-05-06: 0 records
Synced 2024-05-07: 0 records
Synced 2024-05-08: 0 records
Synced 2024-05

In [21]:
#| export
from datetime import date, timedelta


def iter_dates(start_date: str,  # Start date (YYYY-MM-DD)
               end_date: str  # End date (YYYY-MM-DD)
               ) -> list[str]:
    "Generate all dates between start and end inclusive."

    start_d = datetime.strptime(start_date, "%Y-%m-%d").date()
    end_d = datetime.strptime(end_date, "%Y-%m-%d").date()
    return [(start_d + timedelta(days=i)).strftime("%Y-%m-%d")
            for i in range((end_d - start_d).days + 1)]


def get_missing_dates(session: Session,  # Active database session
                      site_url: str,  # GSC property URL
                      start_date: str,  # Start date (YYYY-MM-DD)
                      end_date: str  # End date (YYYY-MM-DD)
                      ) -> list[str]:
    "Return dates in range that have no stored GSC data."
    stored = set(session.exec(
        select(GSCAnalytics.date).where(GSCAnalytics.site_url == site_url).distinct()
    ).all())
    return sorted(set(iter_dates(start_date, end_date)) - stored)




In [ ]:
#| hide
#| eval: false
missing = get_missing_dates(session, "sc-domain:shelid.com", "2026-01-01", "2026-03-28")
print(len(missing), missing[:5])


0 []


In [22]:
#| export
def sync_missing_dates(session: Session, auth: GSCAuth, site_url: str, start_date: str, end_date: str) -> dict:
    "Fetch and store GSC data for missing dates only."
    return _sync_dates(session, auth, site_url, get_missing_dates(session, site_url, start_date, end_date))

In [23]:
# | export
def daily_sync(session: Session,  # Active database session
               auth: GSCAuth,  # Authenticated GSCAuth instance
               sites: list[str]  # List of GSC property URLs to sync
               ) -> dict:
    "Sync missing GSC data for all sites up to today."
    _, end = get_date_range("today")
    return {site: sync_missing_dates(session, auth, site,
                                     min(get_missing_dates(session, site, "2020-01-01", end), default=end),
                                     end)
            for site in sites}

In [25]:
#| hide
#| eval: false
with get_session() as session:
    daily_sync_result = daily_sync(session, auth, ["sc-domain:gpuvec.com"])
    print(daily_sync_result)

Synced 2020-01-01: 0 records
Synced 2020-01-02: 0 records
Synced 2020-01-03: 0 records
Synced 2020-01-04: 0 records
Synced 2020-01-05: 0 records
Synced 2020-01-06: 0 records
Synced 2020-01-07: 0 records
Synced 2020-01-08: 0 records
Synced 2020-01-09: 0 records
Synced 2020-01-10: 0 records
Synced 2020-01-11: 0 records
Synced 2020-01-12: 0 records
Synced 2020-01-13: 0 records
Synced 2020-01-14: 0 records
Synced 2020-01-15: 0 records
Synced 2020-01-16: 0 records
Synced 2020-01-17: 0 records
Synced 2020-01-18: 0 records
Synced 2020-01-19: 0 records
Synced 2020-01-20: 0 records
Synced 2020-01-21: 0 records
Synced 2020-01-22: 0 records
Synced 2020-01-23: 0 records
Synced 2020-01-24: 0 records
Synced 2020-01-25: 0 records
Synced 2020-01-26: 0 records
Synced 2020-01-27: 0 records
Synced 2020-01-28: 0 records
Synced 2020-01-29: 0 records
Synced 2020-01-30: 0 records
Synced 2020-01-31: 0 records
Synced 2020-02-01: 0 records
Synced 2020-02-02: 0 records
Synced 2020-02-03: 0 records
Synced 2020-02

In [26]:
#| export
def compare_date_ranges(session: Session,  # Active database session
                        site_url: str,  # GSC property URL
                        start1: str,  # First period start date (YYYY-MM-DD)
                        end1: str,  # First period end date (YYYY-MM-DD)
                        start2: str,  # Second period start date (YYYY-MM-DD)
                        end2: str,  # Second period end date (YYYY-MM-DD)
                        page_url: str | None = None  # Optional specific page to compare
                        ) -> dict:
    "Compare GSC metrics between two date ranges, optionally for a specific page."
    # Get metrics for period 1
    base = select(
        func.sum(GSCAnalytics.clicks).label("clicks"),
        func.sum(GSCAnalytics.impressions).label("impressions"),
        func.avg(GSCAnalytics.position).label("avg_position"),
        func.avg(GSCAnalytics.ctr).label("avg_ctr")
    )
    filters1 = compose(
        partial(filter_site, site_url=site_url),
        partial(filter_dates, start=start1, end=end1)
    )
    if page_url:
        filters1 = compose(filters1, lambda q: q.where(GSCAnalytics.page == page_url))
    row1 = session.exec(filters1(base)).first()

    # Get metrics for period 2
    filters2 = compose(
        partial(filter_site, site_url=site_url),
        partial(filter_dates, start=start2, end=end2)
    )
    if page_url:
        filters2 = compose(filters2, lambda q: q.where(GSCAnalytics.page == page_url))
    row2 = session.exec(filters2(base)).first()

    def to_dict(row):
        return {
            "clicks": row.clicks or 0,
            "impressions": row.impressions or 0,
            "avg_position": round(row.avg_position or 0, 2),
            "avg_ctr": round((row.avg_ctr or 0) * 100, 2)
        }

    return {
        "page_url": page_url,
        "period1": {"start": start1, "end": end1, **to_dict(row1)},
        "period2": {"start": start2, "end": end2, **to_dict(row2)}
    }

In [27]:
#| export
def get_country_breakdown(session: Session,  # Active database session
                          site_url: str,  # GSC property URL
                          start_date: str,  # Start date (YYYY-MM-DD)
                          end_date: str,  # End date (YYYY-MM-DD)
                          page_url: str | None = None,  # Optional specific page to filter
                          limit: int = 20  # Max number of countries to return
                          ) -> list[dict]:
    "Get traffic metrics grouped by country, optionally for a specific page."
    base = select(
        GSCAnalytics.country,
        func.sum(GSCAnalytics.clicks).label("clicks"),
        func.sum(GSCAnalytics.impressions).label("impressions"),
        func.avg(GSCAnalytics.position).label("avg_position"),
        func.avg(GSCAnalytics.ctr).label("avg_ctr")
    )
    filters = compose(
        partial(filter_site, site_url=site_url),
        partial(filter_dates, start=start_date, end=end_date)
    )
    if page_url:
        filters = compose(filters, lambda q: q.where(GSCAnalytics.page == page_url))

    query = (filters(base)
             .where(GSCAnalytics.country.isnot(None))
             .group_by(GSCAnalytics.country)
             .order_by(func.sum(GSCAnalytics.clicks).desc())
             .limit(limit))

    return [row._asdict() for row in session.exec(query)]